In [ ]:
%dotenv
from massive import RESTClient
from os import environ
import pandas as pd
from pathlib import Path
from collections.abc import Generator
from datetime import date
from urllib3.exceptions import MaxRetryError
from time import sleep
from math import ceil

In [ ]:
# Requires API key in .env file or added to environment
client = RESTClient(api_key=environ["MASSIVE_KEY"])

In [ ]:
# Loads data for a single ticker over some period
# This does notably include after-hours trading, which we may want to filter out
# Rate limit is 5 internal requests per minute, which translates to about 1 ticker-month per minute
def get_stock_data(ticker: str, start_date: str, end_date: str, steps: list[int] = list(range(1,21)) + [60]) -> Generator[tuple[date, pd.DataFrame]]:
    while True:
        try:
            aggs = list(client.list_aggs(ticker, 1, "minute", start_date, end_date))
            break
        except MaxRetryError as e:
            print(f"Rate limited ({e}), retrying in 1 minute")
            inc = 10
            for i in range(ceil(60/inc)):
                sleep(inc)
                print(f"{60-(i+1)*inc} seconds remaining")

    data = pd.DataFrame(aggs)

    # Adds time column (EST, in alignment with actual stock market) for convenience
    data["time"] = pd.to_datetime(data["timestamp"], unit='ms').dt.tz_localize("UTC").dt.tz_convert("US/Eastern")

    for (day, day_data) in data.groupby(data["time"].dt.date):
        # Formats data (collects open/time parameters, adds stepped columns to look back in time according to steps list)
        f_data = day_data[["open", "time"]]
        for step in steps:
            f_data[f"back_{step}"] = f_data["open"].shift(step)
        f_data = f_data.iloc[max(steps):] # clear out NaNs created by steps
        yield (day,f_data)


In [ ]:
for ticker in ["TSLA", "MSFT", "NVDA"]:
    print(f"Getting data for {ticker}")
    Path(f"data/{ticker}").mkdir(parents=True, exist_ok=True)
    for (day, data) in get_stock_data(ticker, "2026-05-01", "2026-05-30"):
        data.to_parquet(f"data/{ticker}/{day}.parquet.gz", compression="gzip")

Getting data for TSLA
Rate limited (HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/TSLA/range/1/minute/2026-05-01/2026-05-30 (Caused by ResponseError('too many 429 error responses'))), retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting data for MSFT
Rate limited (HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/MSFT/range/1/minute/1778257260000/2026-05-30?cursor=bGltaXQ9NTAwMCZzb3J0PWFzYw (Caused by ResponseError('too many 429 error responses'))), retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting data for NVDA
Rate limited (HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/NVDA/range/1/minute/1778240760000/2026-05-30?cursor=b